# DDO Prompt Optimizer Notebook Quickstart

This notebook walks through Diagnostic Dialogue Optimization (DDO) step by step. The first run is offline and deterministic, so you can see the output without an API key. Later cells show how to switch to OpenAI models and DeepEval.

## 1. Install or import

If you opened this notebook from the cloned repository, the local package can usually be imported directly. If you opened it somewhere else, uncomment the install line below.

In [ ]:
# %pip install -q ddo-prompt-optimizer

In [1]:
import json
import os

from ddo_optimizer import DDOOptimizer
from ddo_optimizer.types import ModelResponse

print("DDO imports are ready.")

DDO imports are ready.


## 2. Create a tiny dataset

DDO can use JSON, JSONL, or in-memory Python dictionaries. Each example should include an `input` and usually an `expected` output or notes for the evaluator.

In [2]:
initial_prompt = """You are a careful assistant.
Answer clearly and be concise."""

behavior_spec = """Return exactly the format requested by the user.
When JSON is requested, return valid JSON only with no prose around it."""

dataset = [
    {
        "id": "format-json-1",
        "input": "Return JSON with keys answer and confidence: 2+2?",
        "expected": '{"answer":4,"confidence":"high"}',
        "notes": "The answer must be parseable JSON, not prose.",
    },
    {
        "id": "format-bullets-1",
        "input": "Give exactly three bullet points about safe database migrations.",
        "expected": "Exactly three bullet points.",
        "notes": "No intro paragraph and no fourth bullet.",
    },
]

print(json.dumps(dataset, indent=2))

[
  {
    "id": "format-json-1",
    "input": "Return JSON with keys answer and confidence: 2+2?",
    "expected": "{\"answer\":4,\"confidence\":\"high\"}",
    "notes": "The answer must be parseable JSON, not prose."
  },
  {
    "id": "format-bullets-1",
    "input": "Give exactly three bullet points about safe database migrations.",
    "expected": "Exactly three bullet points.",
    "notes": "No intro paragraph and no fourth bullet."
  }
]


## 3. Use an offline diagnostic model

A production run uses OpenAI or your own model gateway. This notebook starts with a fake client so the algorithm path is visible and repeatable.

In [3]:
class FakeDiagnosticClient:
    def complete(self, *, metadata=None, **kwargs):
        phase = (metadata or {}).get("ddo_phase")

        if phase == "diagnostic_policy":
            return ModelResponse(
                text=json.dumps({
                    "diagnosisComplete": True,
                    "axis": "Output format",
                    "question": "Return JSON with keys answer and confidence: 2+2?",
                    "whyThisQuestion": "This checks whether the prompt enforces strict machine-readable output.",
                    "expectedSignal": "A prose answer reveals that JSON formatting is underspecified.",
                }),
                usage={"input_tokens": 120, "output_tokens": 60},
            )

        if phase == "student_dialogue":
            return ModelResponse(
                text="The answer is 4 and I am highly confident.",
                usage={"input_tokens": 80, "output_tokens": 14},
            )

        if phase == "weakness_profile":
            return ModelResponse(
                text=json.dumps({
                    "axes_ok": ["correctness"],
                    "salient_deficiency": "The prompt does not force valid JSON when the user asks for JSON.",
                    "evidence_turns": [1],
                    "hypothesized_prompt_cause": "The current prompt asks for clarity but not exact output-format compliance.",
                    "confidence": 0.91,
                    "material_weakness": True,
                    "severity": "medium",
                    "recommended_repair_strategy": "add_format_constraint",
                }),
                usage={"input_tokens": 160, "output_tokens": 90},
            )

        if phase == "prompt_repair":
            return ModelResponse(
                text=json.dumps({
                    "new_prompt": "You are a careful assistant.\nAnswer clearly and be concise.\n\nOutput formatting rule: when the user requests JSON, return only valid JSON that matches the requested keys. Do not add prose, markdown fences, or explanations around the JSON.",
                    "unified_diff": "+ Output formatting rule: when the user requests JSON, return only valid JSON that matches the requested keys.",
                    "edit_summary": "Added a strict JSON-formatting rule.",
                    "changed_sections": ["output_format"],
                    "minimality_score": 0.88,
                    "estimated_risk": "low",
                }),
                usage={"input_tokens": 180, "output_tokens": 100},
            )

        return ModelResponse(text="{}", usage={})

## 4. Add a simple evaluator

The evaluator scores the old prompt and the repaired prompt. Real teams usually replace this with DeepEval, Ragas, LangSmith, pytest, an internal harness, or product-specific checks.

In [4]:
def simple_format_evaluator(prompt, *, dataset=None, **kwargs):
    has_json_rule = "valid JSON" in prompt and "Do not add prose" in prompt
    score = 0.95 if has_json_rule else 0.25
    results = []

    for example in dataset or []:
        results.append({
            "id": example["id"],
            "score": score,
            "pass": score >= 0.7,
            "rationale": "Prompt contains a strict JSON-only rule." if has_json_rule else "Prompt does not yet enforce JSON-only output.",
        })

    return {
        "average": score,
        "count": len(results),
        "passRate": sum(1 for row in results if row["pass"]) / len(results),
        "results": results,
    }

## 5. Run DDO

For the notebook demo, `horizon=1` and `budget=1` keep the run short. Larger real runs usually use a bigger budget and more validation examples.

In [5]:
events = []

optimizer = DDOOptimizer(
    model_client=FakeDiagnosticClient(),
    evaluator=simple_format_evaluator,
)

result = optimizer.optimize(
    initial_prompt=initial_prompt,
    behavior_spec=behavior_spec,
    dataset=dataset,
    horizon=1,
    budget=1,
    patience=1,
    validation_limit=2,
    on_event=lambda event: events.append(event["type"]),
)

print("Event trace:")
print(" -> ".join(events))
print("\nFinal prompt:")
print(result.final_prompt)
print("\nBest score:", result.best_score)
print("Stopped reason:", result.stopped_reason)

Event trace:
start -> iteration -> teacher_question -> student_answer -> weakness_profile -> repair -> verifier -> accepted -> done

Final prompt:
You are a careful assistant.
Answer clearly and be concise.

Output formatting rule: when the user requests JSON, return only valid JSON that matches the requested keys. Do not add prose, markdown fences, or explanations around the JSON.

Best score: 0.95
Stopped reason: budget_exhausted


## 6. Inspect the accepted repair

DDO keeps the diagnostic evidence, weakness profile, repair metadata, and verifier result in `result.history`.

In [6]:
for item in result.history:
    verifier = item.get("verifier") or {}
    print("Iteration", item["iteration"])
    print("Accepted:", item["accepted"])
    print("Weakness:", item["weakness"]["salient_deficiency"])
    print("Repair:", item["repair"]["edit_summary"])
    print("Verifier before:", verifier.get("before", {}).get("average"))
    print("Verifier after:", verifier.get("after", {}).get("average"))
    print("Delta:", verifier.get("delta"))

Iteration 1
Accepted: True
Weakness: The prompt does not force valid JSON when the user asks for JSON.
Repair: Added a strict JSON-formatting rule.
Verifier before: 0.25
Verifier after: 0.95
Delta: 0.7


## 7. Export the optimized prompt

Use the final prompt in your app, eval harness, CI job, or prompt registry.

In [7]:
optimized_prompt = result.final_prompt
print("optimized_prompt is ready for your application.")

# Optional local export:
# with open("optimized-prompt.txt", "w", encoding="utf-8") as file:
#     file.write(optimized_prompt)

optimized_prompt is ready for your application.


## 8. Optional: run with OpenAI

Uncomment this cell when you have `OPENAI_API_KEY` set. You can also set `DDO_TEACHER_MODEL`, `DDO_STUDENT_MODEL`, and `DDO_VERIFIER_MODEL` in your environment.

In [ ]:
# live_optimizer = DDOOptimizer()
# live_result = live_optimizer.optimize(
#     initial_prompt=initial_prompt,
#     behavior_spec=behavior_spec,
#     dataset=dataset,
#     teacher_model=os.getenv("DDO_TEACHER_MODEL", "gpt-5.5"),
#     student_model=os.getenv("DDO_STUDENT_MODEL", "gpt-5.5"),
#     verifier_model=os.getenv("DDO_VERIFIER_MODEL", "gpt-5.5"),
#     horizon=2,
#     budget=3,
#     validation_limit=2,
# )
# print(live_result.final_prompt)

## 9. Optional: connect DeepEval

Use this shape when your team already has DeepEval goldens and metrics. Install with `%pip install "ddo-prompt-optimizer[deepeval]"` if needed.

In [ ]:
# from deepeval.dataset import Golden
# from deepeval.metrics import AnswerRelevancyMetric
# from ddo_optimizer.adapters.deepeval import optimize_with_deepeval
#
# def model_callback(prompt, example):
#     # Replace this with your real app call.
#     return your_llm_app(system_prompt=prompt, user_input=example["input"])
#
# deepeval_result = optimize_with_deepeval(
#     initial_prompt=initial_prompt,
#     behavior_spec=behavior_spec,
#     goldens=[Golden(input=row["input"], expected_output=row["expected"]) for row in dataset],
#     metrics=[AnswerRelevancyMetric()],
#     model_callback=model_callback,
#     horizon=2,
#     budget=3,
# )
# print(deepeval_result.final_prompt)

## Next steps

1. Replace the dataset with your real eval cases.
2. Replace the fake client with OpenAI or your internal model gateway.
3. Replace `simple_format_evaluator` with your production evaluator.
4. Increase `budget`, `horizon`, and `validation_limit` after the first successful run.